# 02 — Bronze Ingestion

This notebook lands the raw CSV files into the **Bronze layer** of our Delta Lake.

The Bronze layer is the first stop for all incoming data. The rule here is simple: **land everything as-is, change nothing**. We only add audit columns so we know when the data arrived and where it came from.

This means if something goes wrong downstream, we can always come back to Bronze and reprocess from the original source.

In [ ]:
# Confirm Spark is running
print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')

## Step 1 — Define paths

All three layers live in DBFS. We use Delta format for storage which gives us versioning, ACID transactions, and time travel out of the box.

In [ ]:
RAW_PATH    = 'dbfs:/FileStore/sa-banking-pipeline/raw'
BRONZE_PATH = 'dbfs:/FileStore/sa-banking-pipeline/bronze'
SILVER_PATH = 'dbfs:/FileStore/sa-banking-pipeline/silver'
GOLD_PATH   = 'dbfs:/FileStore/sa-banking-pipeline/gold'

print('Paths defined:')
print(f'  Raw:    {RAW_PATH}')
print(f'  Bronze: {BRONZE_PATH}')
print(f'  Silver: {SILVER_PATH}')
print(f'  Gold:   {GOLD_PATH}')

## Step 2 — Ingest customers CSV → Bronze Delta

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime

# Read raw customers CSV
customers_raw = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{RAW_PATH}/customers.csv')

# Add audit columns — this is all we do at Bronze
customers_bronze = customers_raw \
    .withColumn('_ingested_at', F.current_timestamp()) \
    .withColumn('_source_file', F.lit(f'{RAW_PATH}/customers.csv')) \
    .withColumn('_batch_id', F.lit(datetime.now().strftime('%Y%m%d%H%M%S')))

# Write to Bronze Delta
customers_bronze.write.format('delta').mode('overwrite').save(f'{BRONZE_PATH}/customers')

print(f'Customers Bronze: {customers_bronze.count():,} rows written')
customers_bronze.printSchema()

## Step 3 — Ingest transactions CSV → Bronze Delta

Transactions is our large table (~500k rows). We partition by year and month so future queries that filter by date only scan the relevant files rather than the whole table.

In [ ]:
# Read raw transactions CSV
txn_raw = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{RAW_PATH}/transactions.csv')

# Add audit columns
txn_bronze = txn_raw \
    .withColumn('_ingested_at', F.current_timestamp()) \
    .withColumn('_source_file', F.lit(f'{RAW_PATH}/transactions.csv')) \
    .withColumn('_batch_id', F.lit(datetime.now().strftime('%Y%m%d%H%M%S')))

# Write to Bronze Delta — partition by year and month for query performance
txn_bronze.write.format('delta').mode('overwrite') \
    .partitionBy('segment') \
    .save(f'{BRONZE_PATH}/transactions')

print(f'Transactions Bronze: {txn_bronze.count():,} rows written')

## Step 4 — Verify Bronze tables

In [ ]:
# Read back and verify
cust_check = spark.read.format('delta').load(f'{BRONZE_PATH}/customers')
txn_check  = spark.read.format('delta').load(f'{BRONZE_PATH}/transactions')

print(f'Bronze customers: {cust_check.count():,} rows, {len(cust_check.columns)} columns')
print(f'Bronze transactions: {txn_check.count():,} rows, {len(txn_check.columns)} columns')

print('\nBronze audit columns present:')
for col in ['_ingested_at', '_source_file', '_batch_id']:
    print(f'  ✅ {col}')

In [ ]:
# Show sample
print('Sample Bronze transaction:')
txn_check.select('transaction_id', 'customer_id', 'amount', 'merchant_name', '_ingested_at').show(5, truncate=False)

## ✅ Done

Raw data is now in Bronze Delta tables with full audit trail.

**Next:** Open `03_silver_cleaning.ipynb`